# Spanish Pumped Storage — EDA Overview

**Target:** `es_total_ps` — hourly aggregated Spanish hydro pumped storage production (MW).  
Negative = pumping (consuming electricity to store water). Positive = generation (releasing water through turbines).

Pumped storage operators arbitrage the intraday price spread: they pump during cheap off-peak hours (high renewables / low demand) and generate during expensive peak hours. The dispatch decision is driven by residual demand, price signals, and physical constraints.

**Detailed deep-dives:** [temporal_patterns.ipynb](temporal_patterns.ipynb) | [supply_demand.ipynb](supply_demand.ipynb) | [data_quality.ipynb](data_quality.ipynb)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd

from src.data import load_train, load_test, load_plant_metadata
from src.features import add_time_features
from src.plotting import *

train = load_train()
test = load_test()

print(f"Train: {train.shape[0]:,} hours  ({train['datetime_start'].min().strftime('%Y-%m-%d')} to {train['datetime_start'].max().strftime('%Y-%m-%d')})")
print(f"Test:  {test.shape[0]:,} hours  ({test['datetime_start'].min().strftime('%Y-%m-%d')} to {test['datetime_start'].max().strftime('%Y-%m-%d')})")
print(f"Features: {train.shape[1] - 3} numeric  (excl. id, datetime, target)")

## 1. Target Behaviour

The first thing to establish: how does the target actually behave? We need to understand the distribution, range, and time-series dynamics before looking at any features.

In [ ]:
plot_target(train, rolling_window=168).show()  # 7-day rolling mean

In [ ]:
plot_target_distribution(train).show()

In [ ]:
target = train['es_total_ps']
print("=== Target Summary ===")
print(f"Range:      [{target.min():,.0f}, {target.max():,.0f}] MW")
print(f"Mean:       {target.mean():,.0f} MW  (net pumping bias)")
print(f"Std dev:    {target.std():,.0f} MW")
print()
print(f"Generation: {(target > 0).mean():.1%} of hours  |  mean = {target[target > 0].mean():,.0f} MW")
print(f"Pumping:    {(target < 0).mean():.1%} of hours  |  mean = {target[target < 0].mean():,.0f} MW")
print(f"Idle:       {(target == 0).mean():.1%} of hours")
print()
print("The distribution is bimodal — the system is almost always either pumping or generating, rarely idle.")
print("Pumping is slightly more intense on average (-1,738 MW) than generation (+1,363 MW).")

## 2. The Dispatch Cycle

This is the most important chart. Pumped storage follows a predictable intraday arbitrage cycle:
- **Night (00:00–06:00):** Heavy pumping — electricity is cheap (low demand, baseload running).
- **Morning ramp (07:00–10:00):** Transition to generation as demand rises.
- **Midday (11:00–15:00):** Mixed — solar flood can push prices down, creating pumping opportunity.
- **Evening peak (17:00–21:00):** Maximum generation — demand peaks, solar fades.
- **Late evening (22:00–23:00):** Return to pumping.

The hour × weekday heatmap shows this clearly, including the weekend effect (less extreme swings).

In [ ]:
plot_hourly_heatmap(train).show()

In [ ]:
plot_regime_analysis(train).show()

## 3. Seasonal Variation

The dispatch cycle intensity varies by season: summer months have stronger solar, shifting the midday balance. Winter has higher demand and less solar, sharpening the evening peak.

In [ ]:
plot_monthly_hourly_heatmap(train).show()

## 4. What Drives the Target?

Linear correlations tell part of the story. But the real driver is **residual demand** — the demand left after subtracting wind and solar generation. When residual demand is high, prices rise, and pumped storage generates. When it's low (renewables surplus), they pump.

In [ ]:
plot_target_correlations(train, top_n=20).show()

In [ ]:
plot_residual_demand_profile(train).show()

In [ ]:
# Correlation heatmap: target + key fundamentals
key_cols = [
    'es_total_ps',
    'es_demand_f_d1', 'es_wind_f_d1', 'es_solar_f_d1',
    'es_hydro_ror_f_d1', 'es_hydro_res_f_d1', 'es_hydro_inflow_f_d1',
    'es_temp_f_d1', 'es_gas_market_price_d1', 'eua_price_d1',
    'fr_es_ntc_d1', 'es_fr_ntc_d1',
]
plot_correlation_heatmap(train, columns=key_cols, height=600, width=750).show()

## 5. Key Feature Relationships

Scatter plots for the most important fundamentals. Look for non-linear patterns that simple correlation misses.

In [ ]:
top_features = [
    'es_demand_f_d1', 'es_wind_f_d1', 'es_solar_f_d1',
    'es_hydro_ror_f_d1', 'es_hydro_inflow_f_d1', 'es_gas_market_price_d1',
]
plot_feature_vs_target(train, top_features, ncols=3).show()

## 6. Plant Capacity Constraints

The Spanish pumped storage fleet has 16 units. Total capacity bounds the range of the target. Understanding the asymmetry between generation capacity and pump load helps explain why pumping intensity exceeds generation.

In [ ]:
meta = load_plant_metadata()
total_gen = (meta['capacity_per_unit'] * meta['units']).sum()
total_pump = (meta['pump_load_per_unit'] * meta['pump_units']).sum()
print(f"Total generation capacity: {total_gen:,.0f} MW")
print(f"Total pump load capacity:  {total_pump:,.0f} MW")
print(f"Pump efficiency:           {meta['pump_efficiency'].iloc[0]:.0%}")
print()
print(f"Observed target range: [{target.min():,.0f}, {target.max():,.0f}] MW")
print(f"Capacity utilisation:  gen {target.max()/total_gen:.0%}  |  pump {abs(target.min())/total_pump:.0%}")

plot_plant_capacity(meta).show()

---

## Key Takeaways

1. **Bimodal target** — the system is almost always pumping or generating, almost never idle. This is a classification problem embedded in a regression problem.

2. **Strong intraday cycle** — hour of day is the single most important feature. Night = pump, evening peak = generate. The cycle is modulated by weekday/weekend and season.

3. **Residual demand is the economic driver** — `demand - wind - solar` determines whether the system pumps or generates. When renewables flood the grid, prices drop and plants pump. Features that capture this (demand, wind, solar forecasts) should be central to the model.

4. **Capacity constraints matter** — the target is physically bounded by ~4,500 MW in each direction. Models should respect these bounds.

5. **Weak linear correlations are misleading** — most features show |r| < 0.15 with the target because the relationship is non-linear and regime-dependent. Tree-based models or regime-switching approaches are likely to outperform linear models.

See the detailed notebooks for deeper analysis:
- [temporal_patterns.ipynb](temporal_patterns.ipynb) — intraday arbitrage cycle, seasonal patterns
- [supply_demand.ipynb](supply_demand.ipynb) — merit order, renewables, cross-border, price signals
- [data_quality.ipynb](data_quality.ipynb) — missing values, train/test shift, unavailability data